In [7]:
import pandas as pd

# 파일의 정확한 경로와 확장자(.xlsx)를 지정합니다.
file_path = r'F:\Download\Rawdata.xlsx'

try:
    # 엑셀 파일 로딩 (openpyxl 엔진 사용)
    df = pd.read_excel(file_path, engine='openpyxl')
    print("✅ 엑셀 파일을 성공적으로 불러왔습니다.\n")
    
    # 상위 5개 행 출력
    print(df.head())
    
except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다. '{file_path}' 경로를 확인해주세요.")
except ImportError:
    print("❌ 엑셀 파일을 읽기 위한 'openpyxl' 라이브러리가 설치되어 있지 않습니다.")
    print("💡 터미널(또는 명령 프롬프트)에서 'pip install openpyxl'을 입력하여 설치해주세요.")
except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

✅ 엑셀 파일을 성공적으로 불러왔습니다.

  patient_id 사고 및 질환 항목  연령대  source_count Gender
0    P000001  악성신생물 (암)  10대             1      M
1    P000002  악성신생물 (암)  10대             1      M
2    P000003  악성신생물 (암)  10대             1      M
3    P000004  악성신생물 (암)  10대             1      M
4    P000005  악성신생물 (암)  10대             1      M


In [3]:
import pandas as pd
import numpy as np

# 1. 6개 표준 담보 분류 매핑 함수 정의
def classify_risk_6_standard(item):
    """
    '사고 및 질환 항목'을 6개 표준 담보 카테고리로 분류하는 함수
    """
    if pd.isna(item) or not isinstance(item, str):
        return '기타/미분류'

    # [1] 보상 제외
    exemptions = [
        '고의적 자해 (자살)', 
        '고의적 자해(자살)', 
        '정신활성 물질 사용에 의한 정신 및 행동 장애'
    ]
    
    # [2] 표준_상해사망후유장해 (중대 상해를 유발할 수 있는 외부 요인 사고)
    injuries = [
        '운수 사고 (교통사고)', 
        '불의의 물에 빠짐 (익사)', 
        '유독성 물질에 의한 불의의 중독 및 노출', 
        '가해 (타살)', 
        '연기, 불 및 불꽃에 노출'
    ]
    
    # --- 매핑 로직 ---
    
    # 1. 보상 제외
    if item in exemptions:
        return '보상 제외'A
        
    # 2. 표준_상해사망후유장해
    elif item in injuries:
        return '표준_상해사망후유장해'
        
    # 3. 표준_실손의료비 (도수치료/MRI 등 실손의료비 청구가 잦은 '추락/미끄러짐' 할당)
    elif item == '추락 (미끄러짐)':
        return '표준_실손의료비'
        
    # 4. 표준_배상책임 (데이터 텍스트에 관련 키워드가 있을 경우 필터링)
    elif '배상' in item or '책임' in item or '타인' in item:
        return '표준_배상책임'
        
    # 5. 표준_휴대품손해 (데이터 텍스트에 관련 키워드가 있을 경우 필터링)
    elif '휴대품' in item or '분실' in item or '파손' in item:
        return '표준_휴대품손해'
        
    # 6. 표준_질병사망후유장해 (위 5가지에 해당하지 않는 신생물, 순환기계 질환 등 모든 내부 질환)
    else:
        # '기타(미상)'과 같은 애매한 항목 예외 처리
        if item == '기타(미상)' or item == '미상':
            return '기타/미분류'
        return '표준_질병사망후유장해'


# ==========================================
# 2. 데이터 처리 및 새로운 파일 저장
# ==========================================

# 💡 원본 파일과 저장할 파일의 이름을 Rawdata_Classified.xlsx 로 지정했습니다.
input_file_path = r'F:\Download\Rawdata.xlsx'
output_file_path = r'F:\Download\Rawdata_Classified.xlsx'

try:
    print("⏳ 원본 엑셀 파일을 불러오는 중입니다...")
    df = pd.read_excel(input_file_path, engine='openpyxl')
    
    print("⚙️ 지정해주신 6개 표준 항목으로 데이터를 분류하고 있습니다...")
    # 원본 데이터에 새로운 분류 컬럼 추가
    df['담보_분류_결과'] = df['사고 및 질환 항목'].apply(classify_risk_6_standard)
    
    print(f"💾 분류된 데이터를 '{output_file_path}' 파일로 저장합니다...")
    # 기존 파일이 있다면 덮어쓰고, 없다면 새로 생성합니다.
    df.to_excel(output_file_path, index=False, engine='openpyxl')
    
    print("\n✅ 작업 완료! 6개 항목 분류가 성공적으로 적용 및 저장되었습니다.")
    
    # 6개 항목으로 잘 나뉘었는지 분포 확인
    print(f"\n📊 [6대 표준 담보 분류 결과 분포]")
    print(df['담보_분류_결과'].value_counts())

except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다. '{input_file_path}' 경로를 확인해주세요.")
except KeyError:
    print("❌ 엑셀 파일 내에 '사고 및 질환 항목'이라는 이름의 컬럼이 없습니다.")
except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

⏳ 원본 엑셀 파일을 불러오는 중입니다...
⚙️ 지정해주신 6개 표준 항목으로 데이터를 분류하고 있습니다...
💾 분류된 데이터를 'F:\Download\Rawdata_Classified.xlsx' 파일로 저장합니다...

✅ 작업 완료! 6개 항목 분류가 성공적으로 적용 및 저장되었습니다.

📊 [6대 표준 담보 분류 결과 분포]
담보_분류_결과
표준_질병사망후유장해    56636
보상 제외          12575
표준_상해사망후유장해     2173
표준_실손의료비        1027
Name: count, dtype: int64


In [2]:
import pandas as pd

# 1. 파일 경로 설정 (이전 단계에서 생성한 파일명 사용)
input_file_path = r'F:\Download\Rawdata_Classified.xlsx'
output_file_path = r'F:\Download\Step3_Coverage_Risk_Score_Table_6_Standard.xlsx'

try:
    print("⏳ 데이터를 불러오는 중입니다...")
    df = pd.read_excel(input_file_path, engine='openpyxl')
    
    # '담보_분류_결과' 컬럼 존재 여부 확인
    if '담보_분류_결과' not in df.columns:
        raise ValueError("'담보_분류_결과' 컬럼이 존재하지 않습니다. 먼저 6대 담보 분류 코드를 실행했는지 확인해주세요.")
    
    print("⚙️ 6대 표준 담보별 빈도 및 가중치 스코어를 계산하는 중입니다...")
    
    # 2. 담보 카테고리별 발생 빈도(건수) 집계
    coverage_counts = df['담보_분류_결과'].value_counts().reset_index()
    coverage_counts.columns = ['표준_담보_카테고리', '발생_건수']
    
    # 3. 전체 사고 건수 및 발생 비중(%) 계산
    total_cases = coverage_counts['발생_건수'].sum()
    coverage_counts['발생_비중(%)'] = (coverage_counts['발생_건수'] / total_cases * 100).round(2)
    
    # 4. 가중치 정규화 (Min-Max Scaling 변형)
    # 119 데이터 특성상 빈도가 가장 높은 카테고리(보통 질병)를 최대 가중치 1.0으로 설정
    max_count = coverage_counts['발생_건수'].max()
    coverage_counts['Coverage_Risk_Score'] = (coverage_counts['발생_건수'] / max_count).round(4)
    
    # 5. 스코어가 높은 순으로 정렬 (내림차순)
    final_table = coverage_counts.sort_values(by='Coverage_Risk_Score', ascending=False)
    
    # 6. 결과 터미널 출력
    print("\n✅ [6대 표준 담보별 위험 가중치 스코어 산출 결과]")
    print("-" * 70)
    print(final_table.to_string(index=False))
    print("-" * 70)
    
    # 7. 결과를 새로운 엑셀 파일로 저장
    final_table.to_excel(output_file_path, index=False, engine='openpyxl')
    print(f"\n💾 담보별 스코어 테이블이 성공적으로 저장되었습니다:\n   👉 {output_file_path}")

except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다. '{input_file_path}' 경로를 확인해주세요.")
except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

⏳ 데이터를 불러오는 중입니다...
⚙️ 6대 표준 담보별 빈도 및 가중치 스코어를 계산하는 중입니다...

✅ [6대 표준 담보별 위험 가중치 스코어 산출 결과]
----------------------------------------------------------------------
 표준_담보_카테고리  발생_건수  발생_비중(%)  Coverage_Risk_Score
표준_질병사망후유장해  56636     78.21               1.0000
      보상 제외  12575     17.37               0.2220
표준_상해사망후유장해   2173      3.00               0.0384
   표준_실손의료비   1027      1.42               0.0181
----------------------------------------------------------------------

💾 담보별 스코어 테이블이 성공적으로 저장되었습니다:
   👉 F:\Download\Step3_Coverage_Risk_Score_Table_6_Standard.xlsx


In [3]:
import pandas as pd

# 💡 저장하신 파일명에 맞게 경로를 설정해 주세요.
# 이전 단계에서 'Coverage_Risk_Score_Table_6_Standard.xlsx'로 저장하셨다면 파일명을 수정해주세요.
file_path = r'F:\Download\Step3_Coverage_Risk_Score_Table_6_Standard.xlsx'

try:
    # 엑셀 파일 읽기
    df_score = pd.read_excel(file_path, engine='openpyxl')
    
    print("📊 [6대 표준 담보별 위험 가중치 스코어 표 확인]\n")
    
    # 표 데이터를 터미널에 깔끔하게 전체 출력
    print("-" * 75)
    print(df_score.to_string(index=False))
    print("-" * 75)
    
    print("\n✅ 데이터 로딩 성공! 이 데이터프레임(df_score)을 추천 엔진의 Cost Weight로 바로 활용하시면 됩니다.")

except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다. '{file_path}' 경로와 파일명이 정확한지 확인해주세요.")
    print("💡 팁: 이전 코드에서 파일명을 'Coverage_Risk_Score_Table_6_Standard.xlsx'로 저장했을 수 있습니다.")
except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

📊 [6대 표준 담보별 위험 가중치 스코어 표 확인]

---------------------------------------------------------------------------
 표준_담보_카테고리  발생_건수  발생_비중(%)  Coverage_Risk_Score
표준_질병사망후유장해  56636     78.21               1.0000
      보상 제외  12575     17.37               0.2220
표준_상해사망후유장해   2173      3.00               0.0384
   표준_실손의료비   1027      1.42               0.0181
---------------------------------------------------------------------------

✅ 데이터 로딩 성공! 이 데이터프레임(df_score)을 추천 엔진의 Cost Weight로 바로 활용하시면 됩니다.
